In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load tables
sessions = spark.table("retailnew.silver.clickstream_sessions")
sales = spark.table("retailnew.silver.pos_clean")

# Join using user_id <-> customer_id
joined = (
    sessions.alias("s")
    .join(sales.alias("p"), F.col("s.user_id") == F.col("p.customer_id"), "inner")
)

# Window ordered by event_time (clickstream)
w = Window.partitionBy("transaction_id").orderBy(F.col("s.event_time"))

# Attribution logic
attribution = (
    joined
    .withColumn("touch_order", F.row_number().over(w))
    .withColumn(
        "credit",
        F.when(F.col("touch_order") == 1, 0.4)
         .when(F.col("touch_order") == 2, 0.3)
         .otherwise(0.3)
    )
    .select(
        "p.transaction_id",
        "s.session_id",
        "s.user_id",
        "s.event_time",
        "s.event_type",
        "s.page",
        "s.device",
        "touch_order",
        "credit"
    )
)

attribution.show()

# Write to Gold
(
    attribution.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("retailnew.gold.marketing_attribution")
)
